# Q3: UNet Normal False-Positive Evaluation

Evaluate whether trained UNet checkpoints hallucinate tumor masks on normal BTXRD images. This notebook does not train models; it evaluates existing `best.pt` checkpoints on the full test split with `tumor_only=false`.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess

REPO_URL = "https://github.com/lehngoc/BTXRD-LViT.git"
BRANCH = "model/e1-unet-baseline"
REPO_ROOT = Path("/kaggle/working/BTXRD-LViT")

if not REPO_ROOT.exists():
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(REPO_ROOT)], check=True)
else:
    print(f"Repo already exists: {REPO_ROOT}")

os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT =", REPO_ROOT)

In [ ]:
import json
import platform
import zipfile

import pandas as pd
import torch
import yaml

print("Python:", sys.version)
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Configure Inputs

Set dataset roots and checkpoint artifact paths. `checkpoint_or_artifact` may be a `best.pt` file, a directory containing `best.pt`, or a zip file containing `best.pt`.

In [ ]:
# Change these paths if your Kaggle input names are different.
PREPROCESSED_DATA_ROOT = Path("/kaggle/input/datasets/lehngoc/btxrd-preprocessed-dataset/btxrd-preprocessed")
RAW_DATA_ROOT = Path("/kaggle/input/datasets/lehngoc/btxrd-raw/btxrd-raw")

OUTPUT_DIR = Path("/kaggle/working/q3_unet_normal_fp_eval")
CONFIG_DIR = OUTPUT_DIR / "configs"
METRICS_DIR = OUTPUT_DIR / "metrics"
VIS_DIR = OUTPUT_DIR / "visual_checks"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
RAW_MANIFEST_DIR = OUTPUT_DIR / "raw_manifests"

for path in [OUTPUT_DIR, CONFIG_DIR, METRICS_DIR, VIS_DIR, CHECKPOINT_DIR, RAW_MANIFEST_DIR]:
    path.mkdir(parents=True, exist_ok=True)

# Fill in paths after attaching full artifact zips/checkpoints to the Kaggle notebook.
# Keep enabled=False for checkpoints you do not want to evaluate yet.
RUNS = [
    {
        "name": "E1a_clean_preprocessed_tumor_only",
        "enabled": True,
        "data_mode": "preprocessed",
        "checkpoint_or_artifact": "/kaggle/input/CHANGE-ME-E1A/best.pt",
        "model": {"base_channels": 32},
        "training": {"image_mean": [0.485, 0.456, 0.406], "image_std": [0.229, 0.224, 0.225], "batch_size": 4, "num_workers": 2},
    },
    {
        "name": "E1as_strong_preprocessed_tumor_only",
        "enabled": True,
        "data_mode": "preprocessed",
        "checkpoint_or_artifact": "/kaggle/input/CHANGE-ME-E1AS/best.pt",
        "model": {"base_channels": 64},
        "training": {"image_mean": [0.0, 0.0, 0.0], "image_std": [1.0, 1.0, 1.0], "batch_size": 4, "num_workers": 2},
    },
    {
        "name": "E1b_normal_aware_preprocessed",
        "enabled": True,
        "data_mode": "preprocessed",
        "checkpoint_or_artifact": "/kaggle/input/CHANGE-ME-E1B/best.pt",
        "model": {"base_channels": 32},
        "training": {"image_mean": [0.485, 0.456, 0.406], "image_std": [0.229, 0.224, 0.225], "batch_size": 4, "num_workers": 2},
    },
    {
        "name": "E1bs_strong_normal_aware_preprocessed",
        "enabled": False,
        "data_mode": "preprocessed",
        "checkpoint_or_artifact": "/kaggle/input/CHANGE-ME-E1BS/best.pt",
        "model": {"base_channels": 64},
        "training": {"image_mean": [0.0, 0.0, 0.0], "image_std": [1.0, 1.0, 1.0], "batch_size": 4, "num_workers": 2},
    },
    {
        "name": "E0s_strong_raw_tumor_only_optional",
        "enabled": False,
        "data_mode": "raw",
        "checkpoint_or_artifact": "/kaggle/input/CHANGE-ME-E0S/best.pt",
        "model": {"base_channels": 64},
        "training": {"image_mean": [0.0, 0.0, 0.0], "image_std": [1.0, 1.0, 1.0], "batch_size": 4, "num_workers": 2},
    },
]

THRESHOLDS = [0.3, 0.4, 0.5, 0.6, 0.7]
PRIMARY_THRESHOLD = 0.5
OPTIONAL_VIS_THRESHOLD = 0.7

print("PREPROCESSED_DATA_ROOT =", PREPROCESSED_DATA_ROOT)
print("RAW_DATA_ROOT =", RAW_DATA_ROOT)
print("OUTPUT_DIR =", OUTPUT_DIR)

In [ ]:
required_preprocessed = [
    PREPROCESSED_DATA_ROOT / "data/exports/btxrd_preprocessed/test.csv",
    PREPROCESSED_DATA_ROOT / "data/processed/images_preprocessed",
    PREPROCESSED_DATA_ROOT / "data/processed/masks_preprocessed",
]
required_raw = [
    RAW_DATA_ROOT / "data/exports/btxrd_preprocessed/test.csv",
    RAW_DATA_ROOT / "data/raw/images",
    RAW_DATA_ROOT / "data/processed/masks",
]

needs_raw = any(run["enabled"] and run["data_mode"] == "raw" for run in RUNS)
required = required_preprocessed + (required_raw if needs_raw else [])

for path in required:
    print(path, "->", path.exists())

missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("Missing required dataset inputs:\n" + "\n".join(missing))

## Helper Functions

In [ ]:
def resolve_checkpoint(run: dict) -> Path:
    source = Path(run["checkpoint_or_artifact"])
    if not source.exists():
        raise FileNotFoundError(f"Missing checkpoint/artifact for {run['name']}: {source}")

    if source.is_file() and source.suffix.lower() == ".pt":
        return source

    if source.is_file() and source.suffix.lower() == ".zip":
        extract_dir = CHECKPOINT_DIR / run["name"]
        extract_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(source) as z:
            z.extractall(extract_dir)
        candidates = sorted(extract_dir.rglob("best.pt"))
        if candidates:
            return candidates[0]
        raise FileNotFoundError(f"No best.pt found inside {source}")

    if source.is_dir():
        candidates = sorted(source.rglob("best.pt"))
        if candidates:
            return candidates[0]
        raise FileNotFoundError(f"No best.pt found under {source}")

    raise ValueError(f"Unsupported checkpoint/artifact path: {source}")


def find_relative_raw_image(image_id: str, root: Path) -> str:
    stem = Path(str(image_id)).stem
    candidates = [
        Path("data/raw/images") / str(image_id),
        Path("data/raw/images") / f"{stem}.jpeg",
        Path("data/raw/images") / f"{stem}.jpg",
        Path("data/raw/images") / f"{stem}.png",
    ]
    for rel_path in candidates:
        if (root / rel_path).exists():
            return str(rel_path)
    raise FileNotFoundError(f"Missing raw image for {image_id}")


def find_relative_raw_mask(image_id: str, root: Path) -> str:
    stem = Path(str(image_id)).stem
    candidates = [
        Path("data/processed/masks") / f"{stem}.png",
        Path("data/processed/masks") / f"{stem}.jpg",
        Path("data/processed/masks") / f"{stem}.jpeg",
    ]
    for rel_path in candidates:
        if (root / rel_path).exists():
            return str(rel_path)
    raise FileNotFoundError(f"Missing raw mask for {image_id}")


def build_eval_manifest(run: dict) -> tuple[Path, Path]:
    if run["data_mode"] == "preprocessed":
        return PREPROCESSED_DATA_ROOT, PREPROCESSED_DATA_ROOT / "data/exports/btxrd_preprocessed/test.csv"

    if run["data_mode"] != "raw":
        raise ValueError(f"Unsupported data_mode: {run['data_mode']}")

    src_csv = RAW_DATA_ROOT / "data/exports/btxrd_preprocessed/test.csv"
    dst_csv = RAW_MANIFEST_DIR / f"{run['name']}_test.csv"
    df = pd.read_csv(src_csv)
    df["image_path"] = df["image_id"].map(lambda image_id: find_relative_raw_image(image_id, RAW_DATA_ROOT))
    df["mask_path"] = df["image_id"].map(lambda image_id: find_relative_raw_mask(image_id, RAW_DATA_ROOT))
    df.to_csv(dst_csv, index=False)
    return RAW_DATA_ROOT, dst_csv


def write_runtime_config(run: dict, checkpoint: Path) -> Path:
    data_root, test_csv = build_eval_manifest(run)
    train_cfg = run["training"]
    cfg = {
        "experiment": {"name": f"Q3_{run['name']}"},
        "data": {
            "root_dir": str(data_root),
            "tumor_only": False,
            "train_csv": str(test_csv),
            "val_csv": str(test_csv),
            "test_csv": str(test_csv),
        },
        "model": {
            "name": "unet",
            "in_channels": 3,
            "out_channels": 1,
            "base_channels": int(run["model"]["base_channels"]),
        },
        "training": {
            "seed": 42,
            "device": "cuda" if torch.cuda.is_available() else "cpu",
            "image_size": 224,
            "image_mean": train_cfg["image_mean"],
            "image_std": train_cfg["image_std"],
            "batch_size": int(train_cfg.get("batch_size", 4)),
            "num_workers": int(train_cfg.get("num_workers", 2)),
            "text_column": "text_lvit_prompt",
            "output_dir": str(OUTPUT_DIR / run["name"]),
        },
        "metrics": {"threshold": PRIMARY_THRESHOLD, "min_fp_area_ratio": 0.001},
        "q3": {"checkpoint": str(checkpoint), "data_mode": run["data_mode"]},
    }
    config_path = CONFIG_DIR / f"{run['name']}.yaml"
    config_path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
    return config_path


def run_cmd(cmd: list[str]) -> None:
    print("$", " ".join(str(part) for part in cmd))
    subprocess.run([str(part) for part in cmd], check=True)

## Smoke Load Checkpoints

In [ ]:
from src.data import BTXRDSegmentationDataset
from src.models import UNet

active_runs = []
for run in RUNS:
    if not run["enabled"]:
        print(f"SKIP {run['name']} (enabled=False)")
        continue

    checkpoint = resolve_checkpoint(run)
    config_path = write_runtime_config(run, checkpoint)
    cfg = yaml.safe_load(config_path.read_text())
    dataset = BTXRDSegmentationDataset(
        csv_path=cfg["data"]["test_csv"],
        root_dir=cfg["data"]["root_dir"],
        image_size=cfg["training"]["image_size"],
        image_mean=tuple(cfg["training"]["image_mean"]),
        image_std=tuple(cfg["training"]["image_std"]),
        tumor_only=False,
        max_samples=2,
    )
    normal_count = int((pd.read_csv(cfg["data"]["test_csv"])["tumor"].astype(int) == 0).sum())
    assert normal_count > 0, f"No normal cases in test split for {run['name']}"

    model = UNet(base_channels=cfg["model"]["base_channels"])
    state = torch.load(checkpoint, map_location="cpu")
    model.load_state_dict(state["model_state_dict"])
    sample = dataset[0]
    with torch.no_grad():
        logits = model(sample["image"].unsqueeze(0))
    assert tuple(logits.shape) == tuple(sample["mask"].unsqueeze(0).shape)

    active_runs.append({**run, "checkpoint": checkpoint, "config_path": config_path})
    print(f"OK {run['name']} | checkpoint={checkpoint} | normal_count={normal_count}")

if not active_runs:
    raise RuntimeError("No enabled runs were configured.")

## Evaluate Metrics, Threshold Sweep, And Visuals

In [ ]:
summary_rows = []

for run in active_runs:
    name = run["name"]
    checkpoint = run["checkpoint"]
    config_path = run["config_path"]
    run_metrics_dir = METRICS_DIR / name
    run_vis_dir = VIS_DIR / name
    run_metrics_dir.mkdir(parents=True, exist_ok=True)
    run_vis_dir.mkdir(parents=True, exist_ok=True)

    primary_metrics_path = run_metrics_dir / "test_metrics_thr50.json"
    run_cmd([
        sys.executable,
        "src/training/evaluate_unet.py",
        "--config", config_path,
        "--checkpoint", checkpoint,
        "--split", "test",
        "--threshold", PRIMARY_THRESHOLD,
        "--output", primary_metrics_path,
        "--device", "cuda" if torch.cuda.is_available() else "cpu",
    ])

    sweep_rows = []
    for threshold in THRESHOLDS:
        output_path = run_metrics_dir / f"test_metrics_thr{int(threshold * 100):02d}.json"
        if threshold == PRIMARY_THRESHOLD and output_path == primary_metrics_path:
            metrics = json.loads(primary_metrics_path.read_text())
        else:
            run_cmd([
                sys.executable,
                "src/training/evaluate_unet.py",
                "--config", config_path,
                "--checkpoint", checkpoint,
                "--split", "test",
                "--threshold", threshold,
                "--output", output_path,
                "--device", "cuda" if torch.cuda.is_available() else "cpu",
            ])
            metrics = json.loads(output_path.read_text())

        sweep_rows.append({
            "run": name,
            "data_mode": run["data_mode"],
            "threshold": threshold,
            "normal_count": metrics["normal_count"],
            "normal_pred_area_ratio": metrics["normal_pred_area_ratio"],
            "normal_fp_image_rate": metrics["normal_fp_image_rate"],
            "tumor_dice": metrics["tumor_dice"],
            "tumor_iou": metrics["tumor_iou"],
            "tumor_precision": metrics["tumor_precision"],
            "tumor_recall": metrics["tumor_recall"],
        })

    sweep_path = run_metrics_dir / "test_threshold_sweep_summary.csv"
    pd.DataFrame(sweep_rows).to_csv(sweep_path, index=False)
    summary_rows.extend(sweep_rows)

    for threshold in [PRIMARY_THRESHOLD, OPTIONAL_VIS_THRESHOLD]:
        suffix = f"thr{int(threshold * 100):02d}"
        run_cmd([
            sys.executable,
            "src/training/visualize_unet_predictions.py",
            "--config", config_path,
            "--checkpoint", checkpoint,
            "--split", "test",
            "--threshold", threshold,
            "--output-dir", run_vis_dir / suffix,
            "--max-tumor", 0,
            "--max-normal", 24,
            "--device", "cuda" if torch.cuda.is_available() else "cpu",
        ])

summary_df = pd.DataFrame(summary_rows)
summary_csv = OUTPUT_DIR / "q3_normal_fp_summary.csv"
summary_json = OUTPUT_DIR / "q3_normal_fp_summary.json"
summary_df.to_csv(summary_csv, index=False)
summary_json.write_text(json.dumps(summary_rows, indent=2), encoding="utf-8")

display(summary_df.sort_values(["run", "threshold"]))

## Compact Report Table

In [ ]:
report = summary_df[summary_df["threshold"] == PRIMARY_THRESHOLD].copy()
report = report[[
    "run",
    "data_mode",
    "threshold",
    "normal_count",
    "normal_pred_area_ratio",
    "normal_fp_image_rate",
    "tumor_dice",
    "tumor_iou",
    "tumor_precision",
    "tumor_recall",
]]
report_path = OUTPUT_DIR / "q3_normal_fp_report_thr50.csv"
report.to_csv(report_path, index=False)
display(report.sort_values("normal_fp_image_rate"))

## Package Outputs

In [ ]:
zip_path = Path("/kaggle/working/Q3_unet_normal_fp_eval_outputs.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for path in OUTPUT_DIR.rglob("*"):
        if path.is_file():
            z.write(path, arcname=str(path.relative_to(OUTPUT_DIR)))

print("Saved:", zip_path)
print("Size MB:", zip_path.stat().st_size / 1024 / 1024)